# Lab 2: a multi-agent research system

**A research agent fans out to four searchers at once, then chains its findings into a
formatter agent that writes the report.** That is the whole lab.

Scenario 3, in two stages. Two fields, visual art and music, on two channels, one published
and one internal, make four searchers. Then a second run turns what they found
into a three-section report.

**The chain is the part worth watching.** The formatter is a separate run with an empty
context, so the only thing it will ever know about the research is what the prompt carries
into it. A subagent works the same way; here it is visible rather than asserted.

The corpus is three documents, rigged so each section of the report fills from a different
place:

| Section | What fills it |
|---|---|
| Well established | Visual art, where the published source stands on its own |
| Contested | Music, where the two channels report different figures, because they counted different populations over windows that do not overlap |
| Coverage gaps | The internal channel holds nothing on visual art, so that search runs and matches nothing |

**Sections 1 to 4 make no API call**, so the server, the tool calls and the roster can be
rerun as often as you like before you spend anything. Sections 5 and 6 are one run each,
both capped in turns and in money, both on the smaller model.

Setup, once, in the folder above this one: copy `.env.example` to `.env` and read the notes
at the top of it. The cell below prints which credential it found.

## 1. The brief

Not a list of steps: a goal, the fields that must be covered, and the criteria a finished
report has to meet.

In [ ]:
import labkit

lab = labkit.start(model_env="LAB_MODEL_SMALL", workspace=False, out=True)

import corpus
import report as render
import research_tools as R

BRIEF = {
    "topic": "AI in the creative industries",
    "goal": "Survey how AI is changing the creative industries across both fields below.",
    "fields": corpus.FIELDS,
    "criteria": [
        "Every finding carries a source URL, a verbatim excerpt, a publisher and the period "
        "the source measured.",
        "Where two sources disagree, both values are kept with their publisher and period. "
        "Do not choose between them and do not average them.",
        "A search that ran and matched nothing is reported as a gap with the field and the "
        "channel, not as a failure.",
    ],
}

## 2. The corpus and the research server

**`create_sdk_mcp_server` runs the tools inside this Python process**, so there is no
subprocess and they read the corpus directly. The dict key the server is registered under is
what appears in the tool name, so `{"research": server}` gives `mcp__research__search_web`
and friends.

**The two search tools are the whole of the tool-scoping lesson**, and they are what make the
two searchers different agents rather than the same agent twice.

| Tool | Who gets it | Why it is separate |
|---|---|---|
| `search_web` | the web searcher | the published channel: what the sector says about itself |
| `search_internal` | the document searcher | the internal channel: what one organisation recorded |
| `load_document` | the document searcher | opens one document, and only a URL a search returned |
| `verify_fact` | the formatter | confirms one claim against one source, and nothing else |

`load_document` is the constrained replacement for a generic fetcher. A generic fetcher takes
any URL at all, including one an agent invented, and returns something plausible for it,
which is how a source nobody can check ends up cited in a report.

Below: the six-field refusal contract, and the one tool that enforces it. The other three and
the corpus are in `research_tools.py` and `corpus.py`.

In [ ]:
labkit.show_source(R, "failed", "load_document")

In [ ]:
from claude_agent_sdk import create_sdk_mcp_server

SERVER = create_sdk_mcp_server(name="research", version="1.0.0", tools=R.TOOLS)

labkit.show_table([(doc["field"], doc["channel"], doc["publication_date"],
                    doc["collection_period"], doc["publisher"]) for doc in corpus.DOCS],
                  headers=("field", "channel", "published", "period", "publisher"))

## 3. Call the tools directly

**Watch the second and third calls carefully, because they look similar and are opposites.**

A search that ran and matched nothing is a **success** with an empty list: the corpus
answered, and the answer is that there is nothing there. That is the visual-art gap the
report will carry.

A URL that is not in the corpus is a **failure**, because nothing was answered at all.

Merge the two and an agent either retries a question that was already answered, or reports
an outage as "nothing found".

The refusal prints all six fields. Each one is a question the caller would otherwise have to
guess at.

In [ ]:
# The six fields of the refusal contract, plus the note a successful-but-empty search
# carries. Printing them side by side is what makes the second and third calls comparable.
SHAPE = ("status", "failure_type", "attempted_query", "partial_results",
         "alternative_approaches", "coverage_impact", "note")

for label, tool, arguments in [
    ("search_web for music, the published channel", R.search_web, {"query": "music and audio sessions"}),
    ("search_internal for visual art, which it does not hold", R.search_internal, {"query": "visual art and design"}),
    ("load_document on a URL nobody can check", R.load_document, {"url": "https://ai-creative-institute.test/2026/report"}),
]:
    is_error, payload = await labkit.call_sdk_tool(tool, **arguments)
    labkit.show_payload(payload, label=label, is_error=is_error, keys=SHAPE)
    for hit in payload.get("results", []):
        print(f"  {'results':<24} {hit['title'][:44]:<44} {hit['url']}")
    print()

## 4. The two searchers

**`AgentDefinition` carries the whole design of a role in four fields, and each one does a
different job.**

| Field | Who reads it | What it decides |
|---|---|---|
| `description` | the **research agent**, when choosing whom to delegate to | routing. Say when to use this agent and when not to |
| `prompt` | the **subagent**, once it starts | the detail. It costs nothing until that agent runs |
| `tools` | the runtime | the boundary. Leave it out and the subagent inherits every tool in the run |
| `model` | the runtime | cost per role |

The agent's name is the dict key, not a field, and the fields that cross the wire are
camelCase, because they are shared with the TypeScript SDK. A snake_case keyword raises
`TypeError`, which the cell shows.

**The two are partitioned by channel, not by subject, and the partition is mechanical**: one
holds `search_web`, the other holds `search_internal`, and neither can reach the other's
channel. Partitioning the scope is what stops two subagents doing the same work twice, and
doing it in `tools` rather than in a prompt is what makes it hold. Each field gets one of
each, so two fields means four searchers running at once.

In [ ]:
from claude_agent_sdk import AgentDefinition

CAP = 70

PROVENANCE = ("For every source give the claim, a verbatim excerpt, the URL, the publisher, "
              "the publication date and the period it measured. A source without a URL and a "
              "date is not usable.")

EMPTY = ("If the search ran and matched nothing, say so plainly and name the field and the "
         "channel you searched. An empty result is an answer about the world, not a failure "
         "of the tooling, and it must not be reported as one.")

ROSTER = {
    "web_searcher": AgentDefinition(
        description=("Searches the published channel, trade research and official statistics, "
                     "for one field and reports the sources with their provenance. Use it to "
                     "survey what has been reported publicly. Do not use it for the "
                     "organisation own records."),
        prompt=f"You search the published channel for one field.\n\n"
               f"Call search_web once for the field you were given, and nothing else.\n\n"
               f"{PROVENANCE}\n\n{EMPTY}\n\nAt most {CAP} words.",
        tools=["mcp__research__search_web"],
        model=lab.model,
        maxTurns=2,
    ),
    "document_searcher": AgentDefinition(
        description=("Searches the internal channel, the organisation own reviews, notes and "
                     "ledgers, and opens the document behind the best hit to quote from it. "
                     "Use it when a claim has to rest on internal wording. Do not use it for "
                     "published sector research."),
        prompt=f"You search the internal channel and quote from what you open.\n\n"
               f"Call search_internal once for the field you were given. If it returns a "
               f"hit, call load_document on the best one and quote from what you opened.\n\n"
               f"{PROVENANCE} Every quotation must be verbatim.\n\n{EMPTY}\n\n"
               f"If load_document refuses a URL, read the refusal: it says what to do "
               f"instead. Do not retry it, and do not invent a source you could not open. "
               f"At most {CAP} words.",
        tools=["mcp__research__search_internal", "mcp__research__load_document"],
        model=lab.model,
        maxTurns=3,
    ),
}

# And the shape that does not exist. The community guide's example uses these keywords;
# the dataclass has description, prompt, tools, model, and camelCase for the rest.
try:
    AgentDefinition(name="web_searcher", system_prompt="Search.", allowed_tools=["search_web"])
except TypeError as exc:
    print(f"TypeError: {exc}")

## 5. Stage one: four searchers at once

**Three options on the run look like they do the same job and do not.**

- `tools` is the **session-wide** set of built-in tools, and a subagent's `tools` can only
  choose from it. MCP tools are not built-ins, so it does not gate them at all.
- `allowed_tools` is what runs **without asking**. That is approval, not restriction: it
  never narrows anything.
- Only `AgentDefinition.tools` is a **per-role boundary**.

So the research agent is given the spawning tool and nothing else, and the channel split
holds in the roster.

**The brief asks for all four searchers in one response**, because several spawn calls in a
single response run at the same time and the wait is the slowest of them rather than the sum.
The cell proves it by grouping the calls on the response they arrived in, never per call,
which would look identical either way.

It also asks the research agent to report the findings and stop, rather than write anything
up. Writing up is the next stage, and keeping the two apart is what makes the handover
visible.

`max_budget_usd` is a real stop rather than a warning: crossing it ends the run with an
error instead of a result. It is a safety net set above what a run should cost, never a
target.

In [ ]:
SPAWN = ["Task", "Agent"]
RESEARCH_TOOLS = ["mcp__research__search_web", "mcp__research__search_internal",
                  "mcp__research__load_document"]

researcher = labkit.AgentRunner(
    labkit.options(
        model=lab.model,
        mcp_servers={"research": SERVER},
        system_prompt="You lead a research team. You delegate, and each subagent starts with "
                      "an empty context, so whatever it needs must be in the prompt you send it.",
        tools=SPAWN,                            # availability: delegation and nothing else
        allowed_tools=SPAWN + RESEARCH_TOOLS,   # approval, which is a different thing
        agents=ROSTER,
        max_turns=8,
        max_budget_usd=0.20,
    ),
    strip="mcp__research__",
)

RESEARCH_BRIEF = f"""GOAL
  {BRIEF['goal']}

FIELDS
{chr(10).join('  - ' + field['label'] for field in BRIEF['fields'])}

CRITERIA
{chr(10).join('  - ' + criterion for criterion in BRIEF['criteria'])}

HOW TO WORK
  - For each field, spawn a web_searcher and a document_searcher: four in total, all in the
    same response, so they run at the same time.
  - Then report what every searcher returned, each finding with its claim, verbatim excerpt,
    source URL, publisher, publication date and the period measured, and every empty search
    named with its field and channel.
  - Do not write the report. Another agent does that next, and it will only ever see what
    you hand it."""

research = await researcher.ask(RESEARCH_BRIEF)

render.show_spawns(research)
labkit.show_cost(research, runner=researcher)

FINDINGS = research.reply
print("\nthe findings, which are the only thing the next stage will receive:")
print(FINDINGS[:1200])

## 6. Stage two: chain the findings into a formatter

**The second run starts with an empty context.** It has no roster, it cannot spawn anything,
and its only tool is `verify_fact`, so it can check a finding but cannot go and look for one.

It has not seen the brief, the searchers, or anything the first run did. The only thing it
will ever know about the research is the text in `FINDINGS` that the cell pastes into its
prompt. **That is the rule that governs every delegation in this course**, and chaining two
runs is the plainest way to see it: leave something out of the prompt and it is not there at
all.

`output_format` makes the answer a validated object rather than prose, and it arrives on
`ResultMessage.structured_output`. The harness checks it against the JSON schema and stops
there, so the rules a schema cannot express run in `model_validate`: a finding with no source
URL and no excerpt is not established, and a contested entry needs two values from two
different sources, because two numbers from one publisher are not a disagreement.

**Note what is deliberately not a rule.** Visual art comes back both well established and a
coverage gap, and that is correct rather than a contradiction: the published channel answered
and the internal one did not. A gap names a field **and a channel**, which is why `Gap`
carries both. An earlier draft of this lab forbade the overlap, and the first live run was
refused by its own validator. A contract has to describe the research the run can actually do.

In [ ]:
from pydantic import BaseModel, Field, model_validator


class Claim(BaseModel):
    field_id: str
    claim: str
    excerpt: str
    source_name: str
    source_url: str
    publication_date: str = ""
    collection_period: str = ""


class ContestedValue(BaseModel):
    value: str
    source_name: str
    source_url: str
    collection_period: str = ""


class Contested(BaseModel):
    claim: str
    values: list[ContestedValue]
    possible_explanation: str = ""


class Gap(BaseModel):
    field_id: str
    channel: str = ""
    reason: str


class ResearchReport(BaseModel):
    topic: str
    well_established: list[Claim] = Field(default_factory=list)
    contested: list[Contested] = Field(default_factory=list)
    coverage_gaps: list[Gap] = Field(default_factory=list)

    @model_validator(mode="after")
    def _rules_the_schema_cannot_express(self):
        thin = [c.claim[:40] for c in self.well_established if not (c.source_url and c.excerpt)]
        if thin:
            raise ValueError(f"established claims without a url and an excerpt: {thin}")
        for entry in self.contested:
            # Two values are not a disagreement unless they came from two sources. There
            # is deliberately no rule against a field appearing in both well_established
            # and coverage_gaps: visual art is in both here, because the published
            # channel answered and the internal one did not.
            if len({value.source_url for value in entry.values}) < 2:
                raise ValueError(
                    f"contested entry {entry.claim[:40]!r} does not keep two sources")
        return self

In [ ]:
FORMATTER = """You write a research report from findings you are given.

Confirm a finding against its source with verify_fact before it goes into well_established.

Write three sections and nothing else:
  well_established   findings that stand, each with its verbatim excerpt, URL and date.
  contested          disagreements, every value kept with its publisher and period, plus
                     what might explain the difference. Never choose between them and never
                     average them.
  coverage_gaps      a field and the channel that returned nothing, and why that leaves
                     a gap. A field whose other channel did answer still belongs here.

Every field_id must be one of the ids you were given, spelled exactly as written.

You cannot search. If a finding you would need is missing, record it as a coverage gap."""

formatter = labkit.AgentRunner(
    labkit.options(
        model=lab.model,
        mcp_servers={"research": SERVER},
        system_prompt=FORMATTER,
        tools=[],                                          # it cannot spawn anything
        allowed_tools=["mcp__research__verify_fact"],      # and this is all it can call
        output_format={"type": "json_schema", "schema": ResearchReport.model_json_schema()},
        max_turns=4,
        max_budget_usd=0.10,
    ),
    strip="mcp__research__",
)

written = await formatter.ask(
    f"TOPIC\n  {BRIEF['topic']}\n\n"
    f"FIELD IDS, to be used exactly as written\n"
    f"{chr(10).join('  ' + f['id'] + '   ' + f['label'] for f in BRIEF['fields'])}\n\n"
    f"FINDINGS FROM THE RESEARCH RUN\n\n{FINDINGS}"
)

labkit.show_calls(written)
labkit.show_cost(written, runner=formatter)
print()

report = ResearchReport.model_validate(written.structured_output)
render.show_report(report)

(lab.out / "report.json").write_text(report.model_dump_json(indent=2) + "\n", encoding="utf-8")
print(f"\nwrote {(lab.out / 'report.json').relative_to(lab.dir)}, "
      f"for ${researcher.total + formatter.total:.4f} over two runs")

## What you built

| Stage | Where it was built |
|---|---|
| Research brief | Section 1 |
| MCP server, four tools on two channels | Sections 2 and 3 |
| Six-field failure contract | Sections 2 and 3 |
| Two searchers, partitioned by channel | Section 4 |
| Research agent, four spawns in one response | Section 5 |
| Findings chained into the formatter's prompt | Section 6 |
| ResearchReport: established, contested, gaps | Section 6 |

**Four decisions to carry out of this lab.**

**Given work that splits cleanly, spawn it in one response.** Several spawn calls in a single
response run at the same time, and the wait is the slowest of them rather than the sum.
Spread the same calls across turns and nothing is parallel.

**Given a handover, put the findings in the prompt.** The next agent's context starts empty,
whether it is a subagent or a chained run, so anything you leave out is not there at all.

**Given a subagent that must not do something, reach for `AgentDefinition.tools`.** An
allowlist approves and does not restrict, and `tools` on the run is session-wide; the
per-role list is the only boundary there is.

**Given two sources that disagree, keep both.** A report that resolves a conflict silently
has told the reader less than the research found, and a search that matched nothing is a
finding about the world rather than a fault in the tooling.